# 03 — Movement classification under artist-disjoint evaluation

Trains the classifier network on frozen features and evaluates it on the 20 artist-disjoint splits
used in the paper (Sections IV-B to IV-E).

- **Features** (`FEATURE_SOURCE`): `"resnet50_nb02"` (notebook 02) or a tag written by notebook 02B,
  e.g. `"wikiart_siglip2_so400m_full"`. A list of tags concatenates the full-image vectors
  (feature concatenation, e.g. CLIP ViT-L/14 + DINOv2 = 1792 dimensions).
- **Views** (`FEATURE_VIEW`, multi-view experiments): `"full"`, `"full+crops_mean"` (full image and mean of
  the crops concatenated) or `"all_expand"` (each view is a separate training sample; probabilities are
  averaged per painting at test time).
- **Classifier network:** standardisation (not for ResNet50) → dropout 0.3 → dense 256 (ReLU) →
  dropout 0.3 → softmax; Adam (1e-3), batch size 32 (32 × V for expanded views), at most 40 epochs, early
  stopping on an artist-disjoint inner validation set (10% of the training artists, patience 5), class
  weights inversely proportional to class frequency.
- **Splits** (`SPLIT_MODE="artist_disjoint"`): `StratifiedGroupKFold` with artists as groups;
  (a) 10-fold cross-validation (seed 42) and (b) 10 repetitions in which one fold of an artist-grouped
  10-fold split (seeds 42–51) is the test set (SSS scheme). `SPLIT_MODE="random"` gives the leakage
  measurement of Section V-E.
- **Movement levels** (`LEVEL`): 20 (main analysis), 16 and 12 (Section IV-D).
- **Sensitivity analyses:** `MAX_PER_ARTIST=150` (per-artist cap) and `SINGLE_STYLE_ONLY=True`.

**Environment:** TensorFlow (see `requirements-tf.txt`); training runs on the CPU.

**Output:** `results/cv/classification_cv_results_<RUN_MODE>_<SPLIT_MODE>_<TAG>.xlsx` with the sheets
`Folds` (one row per split), `Summary` and `CM_<config>_<scheme>` (pooled confusion matrices).
The paired tests of Section V-B are computed from the `Folds` sheets with `analysis/paired_tests.py`.

In [ ]:
import importlib, platform
print("Python", platform.python_version())
for mod in ["sklearn", "tensorflow", "numpy", "pandas"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s}", getattr(m, "__version__", "?"))
    except ImportError:
        print(f"{mod:12s}", "not installed")

In [ ]:
import json, time, warnings
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                     StratifiedShuffleSplit, GroupShuffleSplit)
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
import sklearn

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__, "| scikit-learn:", sklearn.__version__)
if sklearn.__version__ != "1.7.2":
    print("⚠ The splits of the paper were generated with scikit-learn 1.7.2; other versions of "
          "StratifiedGroupKFold can give different splits (compare with splits/*.csv.gz).")

## Configuration

In [ ]:
import os
from pathlib import Path

# Project root: the repository folder (notebooks/ is one level below it).
# Override with the environment variable ARTEMIS_ROOT if the data live elsewhere.
PROJECT_ROOT = Path(os.environ.get("ARTEMIS_ROOT", Path.cwd().parent)).resolve()

DATA_DIR      = PROJECT_ROOT / "data"
WIKIART_DIR   = DATA_DIR / "wikiart"
METADATA_DIR  = DATA_DIR / "metadata"
EXCEL_DIR     = PROJECT_ROOT / "results" / "excel"     # auxiliary tables
CV_DIR        = PROJECT_ROOT / "results" / "cv"        # classification results (as in this repository)
SPLITS_DIR    = PROJECT_ROOT / "splits"
EMBEDDING_DIR = PROJECT_ROOT / "cache" / "embeddings"
GPU_FEATURE_DIR = EMBEDDING_DIR / "gpu"
for d in (EXCEL_DIR, CV_DIR, SPLITS_DIR):
    d.mkdir(parents=True, exist_ok=True)

METADATA_FILE           = METADATA_DIR / "artemis_metadata.parquet"
SELECTED_MOVEMENTS_FILE = METADATA_DIR / "selected_movements.json"
NB02_EMBEDDINGS_FILE    = EMBEDDING_DIR / "embeddings.npy"
NB02_INDEX_FILE         = EMBEDDING_DIR / "embedding_index.parquet"

# ---------------- Experimental design ----------------
RUN_MODE   = "full"              # "pilot" (quick test) | "full" (protocol of the paper)
SPLIT_MODE = "artist_disjoint"   # "artist_disjoint" | "random" (leakage measurement)
BALANCE    = "class_weight"      # "class_weight" | "none"
HEAD_HIDDEN = 256
DROPOUT    = 0.3
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
SEED       = 42

# ---------------- Features ----------------
FEATURE_SOURCE = "wikiart_siglip2_so400m_full"   # "resnet50_nb02" | notebook-02B tag | list of tags
# Examples used in the paper:
#   "resnet50_nb02"
#   "wikiart_clip_vitl14_full", "wikiart_clip_vith14_full", "wikiart_dinov2_vitl14_full",
#   "wikiart_dinov3_vitl16_full", "wikiart_siglip2_so400m_full", "wikiart_siglip2_so400m_224_full",
#   "wikiart_convnext_l_in22k_full", "wikiart_convnext_l_clip_full", "wikiart_convnext_l_clip_320_full"
#   multi-view : "wikiart_clip_vitl14_fullcrops6_f0.5" or "wikiart_clip_vitl14_fullcrops4_quad"
#                with FEATURE_VIEW = "full+crops_mean" or "all_expand"
#   concatenation : ["wikiart_clip_vitl14_fullcrops6_f0.5", "wikiart_dinov2_vitl14_full"]
FEATURE_VIEW = "full"            # "full" | "crops_mean" | "full+crops_mean" | "crops_expand" | "all_expand"

# ---------------- Movement level and data selection ----------------
LEVEL = 20                       # 20 (main analysis) | 16 | 12
LEVEL_MAPS = {
    20: {},
    16: {"Analytical_Cubism": "Cubism", "Synthetic_Cubism": "Cubism",
         "Action_painting": "Abstract_Expressionism",
         "New_Realism": "Contemporary_Realism"},
    12: {"Analytical_Cubism": "Cubism", "Synthetic_Cubism": "Cubism",
         "Action_painting": "Abstract_Expressionism",
         "Color_Field_Painting": "Abstract_Expressionism",
         "Pointillism": "Post_Impressionism",
         "New_Realism": None, "Contemporary_Realism": None,     # None = removed
         "Naive_Art_Primitivism": None},
}
MAX_PER_ARTIST = None            # 150 for the per-artist cap (Section V-F)
SINGLE_STYLE_ONLY = False        # True: remove paintings with more than one distinct style label

BASE_TAG = f"L{LEVEL}" + (f"_cap{MAX_PER_ARTIST}" if MAX_PER_ARTIST else "") + ("_single" if SINGLE_STYLE_ONLY else "")
TAG = BASE_TAG

RUN_PLANS = {
    "pilot": dict(sss_repeats=2, kfold_splits=3, max_per_class=300, epochs=8, patience=3),
    "full":  dict(sss_repeats=10, kfold_splits=10, max_per_class=None, epochs=40, patience=5),
}
PLAN = dict(RUN_PLANS[RUN_MODE], archs=["ResNet50"])
ARCHS = {"ResNet50": dict(external=False, standardize=False, preprocess_name="notebook 02 (ResNet50, Keras)")}

VIEW_CHOICES = ("full", "crops_mean", "full+crops_mean", "crops_expand", "all_expand")
EXTERNAL = None
SOURCES = [FEATURE_SOURCE] if isinstance(FEATURE_SOURCE, str) else list(FEATURE_SOURCE)
COMBINED = len(SOURCES) > 1
assert not (COMBINED and "resnet50_nb02" in SOURCES), "Concatenation is only supported for notebook-02B features"
if SOURCES != ["resnet50_nb02"]:
    _infos = []
    for _src in SOURCES:
        _info_f = GPU_FEATURE_DIR / f"{_src}_info.json"
        assert _info_f.exists(), f"Feature specification not found: {_info_f} (run notebook 02B)"
        with open(_info_f, encoding="utf-8") as f:
            _infos.append(json.load(f))
    if COMBINED:
        assert FEATURE_VIEW == "full", "Feature concatenation requires FEATURE_VIEW='full'"
        EXTERNAL = dict(tag="+".join(i["tag"] for i in _infos), model_name="+".join(i["model_name"] for i in _infos),
                        view_names=["full"], shape=[_infos[0]["shape"][0], 1, int(sum(i["shape"][2] for i in _infos))],
                        combined=True, parts=_infos)
    else:
        EXTERNAL = _infos[0]
    assert FEATURE_VIEW in VIEW_CHOICES, f"FEATURE_VIEW must be one of {VIEW_CHOICES}"
    _names = EXTERNAL["view_names"]
    if FEATURE_VIEW not in ("crops_mean", "crops_expand"):
        assert "full" in _names, f"no 'full' view in this feature file ({_names})"
    if FEATURE_VIEW != "full":
        assert any(n.startswith("crop") for n in _names), f"no crops in this feature file ({_names})"
    ALIAS = f"{EXTERNAL['model_name']}-{FEATURE_VIEW.replace('+', '_')}"
    if EXTERNAL.get("crop_mode") == "quadrants" and FEATURE_VIEW != "full":
        ALIAS += "-quad"
    ARCHS[ALIAS] = dict(external=True, standardize=True, preprocess_name=f"external:{EXTERNAL['tag']}")
    PLAN = dict(PLAN, archs=[ALIAS])
    TAG = TAG + f"_{ALIAS}"
    print(f"Features: {EXTERNAL['tag']}  (view={FEATURE_VIEW}, D={EXTERNAL['shape'][2]})")

RESULTS_FILE = CV_DIR / f"classification_cv_results_{RUN_MODE}_{SPLIT_MODE}_{TAG}.xlsx"

tf.keras.utils.set_random_seed(SEED)
print(f"RUN_MODE={RUN_MODE}  SPLIT_MODE={SPLIT_MODE}  LEVEL={LEVEL}  MAX_PER_ARTIST={MAX_PER_ARTIST}  "
      f"SINGLE_STYLE_ONLY={SINGLE_STYLE_ONLY}  TAG={TAG}")

## Data: selected movements, movement level and optional filters

In [ ]:
with open(SELECTED_MOVEMENTS_FILE, encoding="utf-8") as f:
    SELECTED = json.load(f)["movements"]

meta = pd.read_parquet(METADATA_FILE)
cols = ["filename", "movement", "artist"] + (["genre"] if "genre" in meta.columns else [])
df = meta[meta["movement"].isin(SELECTED)][cols].copy()
df["path"] = [str(WIKIART_DIR / f) for f in df["filename"]]
exists = np.array([Path(p).is_file() for p in df["path"]])
if not exists.all():
    print(f"⚠ {int((~exists).sum())} images not found on disk; removed.")
df = df[exists].reset_index(drop=True)

if EXTERNAL is not None:        # keep images that have features in every source
    _ok = None
    for _src in SOURCES:
        _idx = pd.read_parquet(GPU_FEATURE_DIR / f"{_src}_index.parquet")
        _s = set(_idx.loc[_idx["load_ok"], "filename"])
        _ok = _s if _ok is None else (_ok & _s)
    df = df[df["filename"].isin(_ok)].reset_index(drop=True)

df["artist"] = df["artist"].fillna("Unknown").astype(str)   # no missing artist names in WikiArt
df["movement_orig"] = df["movement"]
print(f"Start (20 selected movements): {len(df):,} paintings")

if SINGLE_STYLE_ONLY:
    import ast
    def _n_styles(s):
        try:
            return len(set(ast.literal_eval(s)))
        except Exception:
            return 1
    multi = df["genre"].map(_n_styles) > 1
    print(f"SINGLE_STYLE_ONLY: {int(multi.sum()):,} multi-style paintings removed")
    df = df[~multi].reset_index(drop=True)

lm = LEVEL_MAPS[LEVEL]
if lm:
    df["movement"] = df["movement"].map(lambda m: lm.get(m, m))
    n_drop = int(df["movement"].isna().sum())
    df = df.dropna(subset=["movement"]).reset_index(drop=True)
    print(f"LEVEL={LEVEL}: {df['movement'].nunique()} classes ({n_drop:,} paintings removed)")

if MAX_PER_ARTIST:       # shuffle, then keep the first N per (movement, artist): deterministic
    n_before = len(df)
    df = (df.sample(frac=1, random_state=SEED).groupby(["movement", "artist"]).head(MAX_PER_ARTIST)
            .sort_index().reset_index(drop=True))
    print(f"MAX_PER_ARTIST={MAX_PER_ARTIST}: {n_before:,} -> {len(df):,} paintings")

if PLAN["max_per_class"]:   # pilot mode only
    df = (df.sample(frac=1, random_state=SEED).groupby("movement").head(PLAN["max_per_class"])
            .sort_index().reset_index(drop=True))

CLASS_NAMES = sorted(df["movement"].unique())
df["label"] = df["movement"].map({c: i for i, c in enumerate(CLASS_NAMES)})
NUM_CLASSES = len(CLASS_NAMES)

info = df.groupby("movement").agg(paintings=("path", "size"), artists=("artist", "nunique")).sort_values("paintings")
print(f"Classes: {NUM_CLASSES}   Paintings: {len(df):,}   Artists: {df['artist'].nunique():,}")
info

## Splits

In [ ]:
def make_splits(df, scheme, n, seed=SEED):
    """Returns [(train_idx, test_idx), ...]."""
    X = np.zeros(len(df)); y = df["label"].to_numpy(); g = df["artist"].to_numpy()
    out = []
    if scheme == "sss":      # one test fold of a 10-fold split, new seed in every repetition
        for r in range(n):
            if SPLIT_MODE == "artist_disjoint":
                sp = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed + r)
                out.append(next(iter(sp.split(X, y, g))))
            else:
                sp = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=seed + r)
                out.append(next(iter(sp.split(X, y))))
    elif scheme == "kfold":
        if SPLIT_MODE == "artist_disjoint":
            out = list(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed).split(X, y, g))
        else:
            out = list(StratifiedKFold(n_splits=n, shuffle=True, random_state=seed).split(X, y))
    else:
        raise ValueError(scheme)
    if SPLIT_MODE == "artist_disjoint":
        for tr, va in out:
            assert not (set(g[tr]) & set(g[va])), "artist leakage!"
    return out


def inner_split(tr_idx, seed):
    """Artist-disjoint inner validation set (10% of the training artists) for early stopping."""
    g = df["artist"].to_numpy()[tr_idx]
    sp = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=seed)
    a, b = next(iter(sp.split(np.zeros(len(tr_idx)), groups=g)))
    return tr_idx[a], tr_idx[b]


def class_weights(y):
    if BALANCE != "class_weight":
        return None
    cnt = np.bincount(y, minlength=NUM_CLASSES).astype(float)
    w = np.where(cnt > 0, len(y) / (NUM_CLASSES * np.maximum(cnt, 1)), 0.0)
    return {i: float(w[i]) for i in range(NUM_CLASSES)}

## Export the splits

Writes one column per split (`kfold_01` … `kfold_10`, `sss_01` … `sss_10`) with the role of every
painting: `train` (classifier training), `val` (inner validation for early stopping) or `test`.
The files in `splits/` (gzip-compressed CSV) of this repository were produced with this cell. Check: at `LEVEL=20`,
`kfold_01` contains 5,538 test paintings.

In [ ]:
out = pd.DataFrame({"filename": df["filename"].to_numpy(), "movement": df["movement"].to_numpy(),
                    "artist": df["artist"].to_numpy()})
for scheme, n in [("kfold", PLAN["kfold_splits"]), ("sss", PLAN["sss_repeats"])]:
    for i, (tr, va) in enumerate(make_splits(df, scheme, n)):
        fit_idx, es_idx = inner_split(tr, SEED + 1000 * i)
        col = np.full(len(df), "", dtype=object)
        col[fit_idx], col[es_idx], col[va] = "train", "val", "test"
        out[f"{scheme}_{i + 1:02d}"] = col
split_file = SPLITS_DIR / f"splits_{SPLIT_MODE}_{BASE_TAG}.csv.gz"
out.to_csv(split_file, index=False)
print("✓", split_file.name, out.shape, "| kfold_01 test:", int((out["kfold_01"] == "test").sum()))

## Optional checks: inner-validation coverage, majority-class baseline, SSS test counts

In [ ]:
rows = []
for scheme, n in [("kfold", 10), ("sss", 10)]:
    for i, (tr, va) in enumerate(make_splits(df, scheme, n)):
        fit_idx, es_idx = inner_split(tr, SEED + 1000 * i)
        missing = sorted(set(range(NUM_CLASSES)) - set(df["label"].to_numpy()[es_idx]))
        rows.append(dict(scheme=scheme, split=i + 1, n_test=len(va), n_inner_val=len(es_idx),
                         movements_in_inner_val=NUM_CLASSES - len(missing),
                         missing_movements=", ".join(CLASS_NAMES[j] for j in missing)))
cov = pd.DataFrame(rows)
print("Splits with at least one movement missing from the inner validation set:",
      int((cov.movements_in_inner_val < NUM_CLASSES).sum()), "/ 20")
cov.to_excel(EXCEL_DIR / f"inner_validation_coverage_{BASE_TAG}.xlsx", index=False)

y_all = df["label"].to_numpy()
rows = []
for scheme, n in [("kfold", 10), ("sss", 10)]:
    for i, (tr, va) in enumerate(make_splits(df, scheme, n)):
        cls = np.bincount(y_all[tr], minlength=NUM_CLASSES).argmax()
        pred = np.full(len(va), cls)
        rows.append(dict(scheme=scheme, split=i + 1, predicted=CLASS_NAMES[cls],
                         accuracy=float((pred == y_all[va]).mean()),
                         balanced_accuracy=float(balanced_accuracy_score(y_all[va], pred))))
print("\nMajority-class baseline (predict the most frequent training movement):")
print(pd.DataFrame(rows).groupby("scheme")[["accuracy", "balanced_accuracy"]].agg(["mean", "std"]).round(4))

art = df["artist"].to_numpy()
spl = make_splits(df, "sss", 10)
rows = []
for c, name in enumerate(CLASS_NAMES):
    idx = np.concatenate([va[y_all[va] == c] for _, va in spl])
    rows.append(dict(movement=name, test_predictions=len(idx), unique_paintings=len(np.unique(idx)),
                     unique_artists=len(np.unique(art[idx])),
                     repetitions_present=int(sum((y_all[va] == c).any() for _, va in spl))))
pd.DataFrame(rows).to_excel(EXCEL_DIR / f"sss_test_counts_{BASE_TAG}.xlsx", index=False)

## Features and classifier network

In [ ]:
def build_head(input_shape, adapt_data=None):
    inp = layers.Input(shape=input_shape)
    x = inp
    if adapt_data is not None:      # standardisation fitted on the training samples, stored in the model
        norm = layers.Normalization(axis=-1, name="standardize")
        norm.adapt(adapt_data)
        x = norm(x)
    x = layers.Dropout(DROPOUT)(x)
    if HEAD_HIDDEN:
        x = layers.Dense(HEAD_HIDDEN, activation="relu")(x)
        x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def _source_rows(src):
    stem = GPU_FEATURE_DIR / src
    F = np.load(f"{stem}.npy", mmap_mode="r")
    idx = pd.read_parquet(f"{stem}_index.parquet")
    rows = pd.Series(idx["embedding_index"].to_numpy(), index=idx["filename"]).reindex(df["filename"]).to_numpy()
    assert not pd.isnull(rows).any(), f"some paintings have no {src} features"
    return F, rows.astype(int)


def load_external_views():
    """Notebook-02B features as (N, V, D) float32 in the row order of df, with the views of FEATURE_VIEW.
    Concatenation: the L2-normalised full-image vectors of the sources are joined -> (N, 1, D1+D2+...)."""
    if EXTERNAL.get("combined"):
        parts = []
        for src, inf in zip(SOURCES, EXTERNAL["parts"]):
            F, rows = _source_rows(src)
            parts.append(np.asarray(F[rows, inf["view_names"].index("full")]).astype(np.float32))
        return np.ascontiguousarray(np.concatenate(parts, axis=-1)[:, None, :])
    F, rows = _source_rows(FEATURE_SOURCE)
    names = EXTERNAL["view_names"]
    full_i = [i for i, n in enumerate(names) if n == "full"]
    crop_i = [i for i, n in enumerate(names) if n.startswith("crop")]
    sel = {"full": full_i, "crops_mean": crop_i, "full+crops_mean": full_i + crop_i,
           "crops_expand": crop_i, "all_expand": full_i + crop_i}[FEATURE_VIEW]
    A = np.asarray(F[rows])[:, sel].astype(np.float32)
    unit = lambda x: x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-8, None)
    if FEATURE_VIEW == "crops_mean":
        A = unit(A.mean(1, keepdims=True))
    elif FEATURE_VIEW == "full+crops_mean":
        A = np.concatenate([A[:, 0], unit(A[:, 1:].mean(1))], axis=-1)[:, None, :]
    return np.ascontiguousarray(A)


def load_resnet50_features():
    """ResNet50 features of notebook 02 (float32, not L2-normalised), in the row order of df."""
    idx = pd.read_parquet(NB02_INDEX_FILE)
    E = np.load(NB02_EMBEDDINGS_FILE, mmap_mode="r")
    rows = pd.Series(idx["embedding_index"].to_numpy(), index=idx["filename"]).reindex(df["filename"]).to_numpy()
    assert not np.isnan(rows.astype(float)).any(), "some paintings have no notebook-02 features"
    return np.asarray(E[rows.astype(int)], dtype=np.float32)


VIEWS_CACHE = {}
def get_views(arch):
    """(N, V, D): V = 1 except for expanded multi-view conditions, where every view is a training sample."""
    if arch not in VIEWS_CACHE:
        VIEWS_CACHE[arch] = load_external_views() if ARCHS[arch]["external"] else load_resnet50_features()[:, None, :]
        print(f"✓ {arch}: features {VIEWS_CACHE[arch].shape}")
    return VIEWS_CACHE[arch]

## Training and evaluation

In [ ]:
def fit_predict(arch, tr_idx, va_idx, seed):
    fit_idx, es_idx = inner_split(tr_idx, seed)
    y_all = df["label"].to_numpy()
    cw = class_weights(y_all[fit_idx])
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PLAN["patience"], restore_best_weights=True)
    F = get_views(arch)
    V, D = F.shape[1], F.shape[2]
    Xf, yf = F[fit_idx].reshape(-1, D), np.repeat(y_all[fit_idx], V)
    Xe, ye = F[es_idx].reshape(-1, D), np.repeat(y_all[es_idx], V)
    model = build_head((D,), adapt_data=Xf if ARCHS[arch]["standardize"] else None)
    hist = model.fit(Xf, yf, validation_data=(Xe, ye), epochs=PLAN["epochs"], batch_size=BATCH_SIZE * V,
                     class_weight=cw, callbacks=[es], verbose=0)
    pv = model.predict(F[va_idx].reshape(-1, D), batch_size=1024, verbose=0)
    probs = pv.reshape(len(va_idx), V, -1).mean(axis=1)      # average over the views of each painting
    return probs, int(np.argmin(hist.history["val_loss"]) + 1)


def evaluate(y_true, probs):
    y_pred = probs.argmax(1)
    labels = np.unique(y_true)     # macro measures over the classes present in the test fold
    top3 = np.mean([y in np.argsort(p)[-3:] for y, p in zip(y_true, probs)])
    return dict(accuracy=float(np.mean(y_pred == y_true)), top3_accuracy=float(top3),
                balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
                macro_f1=float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
                classes_in_val=int(len(labels)))

In [ ]:
fold_rows, confusions = [], {}
y_all = df["label"].to_numpy()
for arch in PLAN["archs"]:
    for scheme, n in [("sss", PLAN["sss_repeats"]), ("kfold", PLAN["kfold_splits"])]:
        cm_total = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
        for i, (tr, va) in enumerate(tqdm(make_splits(df, scheme, n), desc=f"{arch} / {scheme}")):
            t0 = time.time()
            probs, best_ep = fit_predict(arch, tr, va, SEED + 1000 * i)
            cm_total += confusion_matrix(y_all[va], probs.argmax(1), labels=np.arange(NUM_CLASSES))
            fold_rows.append(dict(arch=arch, scheme=scheme, split=i + 1, n_train=len(tr), n_val=len(va),
                                  best_epoch=best_ep, seconds=round(time.time() - t0, 1),
                                  **evaluate(y_all[va], probs)))
        confusions[(arch, scheme)] = cm_total
folds_df = pd.DataFrame(fold_rows)
folds_df.groupby(["arch", "scheme"])[["accuracy", "balanced_accuracy", "macro_f1", "top3_accuracy"]].mean().round(4)

## Summary with approximate Nadeau–Bengio confidence intervals

Standard error `s · sqrt(1/k + mean(n_test / n_train))`, where `s` is the standard deviation of the
split scores, `k` the number of splits and `n_train` the size of the outer training set (Section IV-C).

In [ ]:
def nb_ci95(x, n_test, n_train):
    x = np.asarray(x, float); k = len(x)
    if k < 2:
        return np.nan, np.nan
    se = x.std(ddof=1) * np.sqrt(1 / k + np.mean(np.asarray(n_test) / np.asarray(n_train)))
    h = stats.t.ppf(0.975, k - 1) * se
    return x.mean() - h, x.mean() + h

rows = []
for (arch, scheme), g in folds_df.groupby(["arch", "scheme"]):
    for mt in ["accuracy", "top3_accuracy", "balanced_accuracy", "macro_f1"]:
        lo, hi = nb_ci95(g[mt], g["n_val"], g["n_train"])
        rows.append(dict(arch=arch, scheme=scheme, metric=mt, mean=g[mt].mean(), std=g[mt].std(ddof=1),
                         nb_ci95_low=lo, nb_ci95_high=hi, n=len(g)))
summary_df = pd.DataFrame(rows)

with pd.ExcelWriter(RESULTS_FILE, engine="openpyxl") as w:
    folds_df.to_excel(w, sheet_name="Folds", index=False)
    summary_df.to_excel(w, sheet_name="Summary", index=False)
    for (arch, scheme), cm in confusions.items():
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_excel(w, sheet_name=f"CM_{arch[:20]}_{scheme}"[:31])
print("✓", RESULTS_FILE.name)
summary_df[summary_df.scheme == "kfold"].round(4)